# Recitation 5: Clustering & PCA with scikit-learn
### Data Science for Entrepreneurship-Economy | Spring 2026

**Today's goal**: Discover hidden customer segments in unlabeled data — from raw features to named, actionable groups visualized in 2D.

By the end of this session you'll be able to:
- Scale features and run K-Means clustering
- Choose K using the Elbow method and Silhouette score
- Profile and name clusters with business meaning
- Use PCA to reduce dimensions and visualize clusters in 2D
- Combine PCA + K-Means into a full segmentation pipeline

**Time**: ~90 minutes | **Discussions**: 6 | **Exercises**: 1 | **Format**: Guided walkthrough with instructor-led discussions

**The big shift**: Every model so far had a target column (churn, price, valuation). Today — NO target. The algorithm finds structure on its own.

### 📖 Quick Recap

- In Recitations 3–4 you built classifiers and regressors — supervised learning with labeled targets
- In Lecture 5 you learned K-Means, Silhouette scores, PCA, and cluster profiling — conceptually
- Today you run the full unsupervised pipeline hands-on for the first time
- Key API change: `.fit(X)` only — no `y`, no `.predict()`, no "right answer"
- Scaling is mandatory (same as KNN in Recitation 3)

So far, someone told the algorithm the answers. Today, the algorithm discovers them.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# scikit-learn imports for unsupervised learning
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

sns.set_style("whitegrid")

In [ ]:
df = pd.read_csv("telebari_subscribers.csv")

### 📖 Meet the Business

> **TeleBari** is an Israeli mobile telecom company with ~2 million subscribers. The company offers a range of plans but has never formally segmented its customer base — everyone gets the same marketing emails, the same retention offers, and the same support priority. The Head of Customer Strategy, Amit, believes the subscriber base contains distinct behavioral groups and wants data-driven segments to personalize plan recommendations, target retention campaigns, and allocate support resources. He exported a sample of 500 subscribers with usage data from the last 3 months.

Amit wants segments — not guesses. Let's find them.

In [ ]:
print(f"Dataset: {df.shape[0]} subscribers x {df.shape[1]} columns\n")
print(df.describe().round(1))

Notice there's NO target column. No `churned`, no `price`, no labels. Just behavioral features. This is unsupervised learning.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
features = [c for c in df.columns if c != "subscriber_id"]
for i, feat in enumerate(features):
    ax = axes[i // 4, i % 4]
    df[feat].hist(bins=20, ax=ax, color="#214073", edgecolor="white")
    ax.set_title(feat, fontsize=9)
plt.tight_layout()
plt.suptitle("Feature Distributions — TeleBari Subscribers", y=1.02, fontsize=13)
plt.show()

Key observation: `monthly_data_gb` has a much wider range than other features. If we don't scale, this one feature will dominate all distance calculations.

In [ ]:
corr = df.drop(columns="subscriber_id").corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, square=True)
plt.title("Feature Correlations — TeleBari")
plt.tight_layout()
plt.show()

Notice: `monthly_minutes` and `international_minutes` are correlated — people who call a lot also call internationally. PCA will compress this shared information later.

In [ ]:
# Drop ID, keep only features
X = df.drop(columns="subscriber_id")

# Scale — CRITICAL for K-Means and PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

print("Before scaling — monthly_data_gb std:", f"{X['monthly_data_gb'].std():.1f}")
print("After scaling  — monthly_data_gb std:", f"{X_scaled['monthly_data_gb'].std():.2f}")
print("\nAll features now have mean ~ 0 and std ~ 1. Ready for clustering.")

---
## 🎯 Section 2: K-Means Clustering
*Let the algorithm find groups. You tell it how many — it figures out where.*

---

### 📖 K-Means Recap

K-Means partitions data into exactly K groups by repeating: assign each point to the nearest centroid → move centroids to the mean of their group → repeat until stable. You choose K. The algorithm finds the best centroids. Always on scaled data.

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
kmeans.fit(X_scaled)

# Add cluster labels to the DataFrame
df["cluster"] = kmeans.labels_

print("Cluster sizes:")
print(df["cluster"].value_counts().sort_index())
print(f"\nInertia: {kmeans.inertia_:.0f}")

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df["monthly_data_gb"], df["monthly_minutes"],
            c=df["cluster"], cmap="Set2", alpha=0.6,
            edgecolor="gray", s=40)
plt.xlabel("Monthly Data (GB)")
plt.ylabel("Monthly Voice (minutes)")
plt.title("Clusters: Data Usage vs. Voice Minutes")
plt.colorbar(label="Cluster")
plt.show()

Even in this 2D slice (2 of 8 features), structure is visible. The upper-left cluster (high voice, low data) looks different from the lower-right (high data, low voice). But 2D shows only part of the story — we'll use PCA later for the full picture.

### ⚠️ Common Mistake

We chose K=4 as a starting guess. But how do we KNOW 4 is the right number? Maybe 3 is better, or 5. Next section: let the data guide the choice.

---
## 🔍 Section 3: Choosing K
*Too few groups = too vague. Too many = chaos. Find the sweet spot.*

---

In [ ]:
inertias = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.plot(K_range, inertias, "bo-", linewidth=2, markersize=8)
plt.xlabel("K (number of clusters)")
plt.ylabel("Inertia (within-cluster distance)")
plt.title("Elbow Method — TeleBari")
plt.show()

In [ ]:
sil_scores = []
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    sil_scores.append(sil)
    print(f"K={k}: Silhouette = {sil:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(K_range, inertias, "bo-", linewidth=2)
axes[0].set_xlabel("K")
axes[0].set_ylabel("Inertia")
axes[0].set_title("Elbow Method")
axes[0].axvline(x=4, color="red", linestyle="--", alpha=0.7)
axes[1].plot(K_range, sil_scores, "gs-", linewidth=2)
axes[1].set_xlabel("K")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Score")
axes[1].axvline(x=4, color="red", linestyle="--", alpha=0.7)
plt.suptitle("Both Methods Point to K=4", fontsize=13)
plt.tight_layout()
plt.show()

### 💬 Discussion 1: Read the Plots

*Instructor leads — students respond:*

- Look at the elbow plot: at which K does the curve "bend" (inertia stops dropping sharply)?
- Look at the silhouette plot: which K has the highest score?
- Do both methods agree? What K would you recommend to Amit?
- Amit says: "My marketing team can realistically manage 3 campaigns. Is K=3 OK?" What do you tell him?

**Takeaway**: Both methods point to **K=4** — the elbow bends at 4 and silhouette peaks at K=4. For Amit's question: K=3 is mathematically worse, but if the team can only handle 3 campaigns, merge the two most similar segments. **Business constraints override math** — but document the tradeoff. K is a business decision aided by math, not purely a math decision.

### 💡 Insight

Math suggests K=4. The business can handle K=3 or K=4. The right answer is: try both, profile both, and let Amit choose based on which segments are most actionable. Data informs — humans decide.

In [ ]:
# Confirmed: K=4
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)

print("Final cluster sizes:")
print(df["cluster"].value_counts().sort_index())

---
## 👥 Section 4: Cluster Profiling
*Numbers are useless until you name them. Turn cluster 0, 1, 2, 3 into "Data Junkies" and "Traditional Callers."*

---

### 📖 Profiling Recap

K-Means gave us 4 groups labeled 0, 1, 2, 3. But "0" means nothing to Amit. Profiling computes the average feature values per cluster so we can understand WHO is in each group and WHAT makes them different. Then we name them.

In [ ]:
# Compute mean of each feature per cluster (on ORIGINAL scale, not scaled)
profiles = df.groupby("cluster")[X.columns.tolist()].mean().round(1)
profiles["size"] = df["cluster"].value_counts().sort_index()
print(profiles.to_string())

In [ ]:
# Normalize profiles for visualization (z-score per feature)
prof_data = profiles.drop(columns="size")
profiles_z = (prof_data - prof_data.mean()) / prof_data.std()

plt.figure(figsize=(12, 4))
sns.heatmap(profiles_z, annot=True, fmt=".1f", cmap="RdYlGn",
            center=0, linewidths=1,
            yticklabels=[f"Cluster {i}" for i in range(4)])
plt.title("Cluster Profiles (z-scored) — Red = Below Average, Green = Above Average")
plt.tight_layout()
plt.show()

### 📖 How to Read the Heatmap

Each row is a cluster. Each column is a feature. Green = that cluster is ABOVE average on this feature. Red = BELOW average. The pattern of greens and reds tells you the personality of each cluster.

### 💬 Discussion 2: Name the Segments

*Instructor leads — students respond:*

- Look at the profile table and heatmap
- For each cluster (0, 1, 2, 3), suggest a **business-friendly name** based on its dominant characteristics
- For each, describe in **one sentence** who this subscriber is
- Hint: look for the most extreme (green/red) values in each row — those define the cluster's identity

**Takeaway**: The four profiles map to:

- **Data Junkies** — High data usage, high app sessions, low voice/SMS. Digital-first subscribers who stream and browse.
- **Traditional Callers** — High voice minutes, high international, high SMS, low data, low app usage. Voice-centric, long-tenure subscribers.
- **Budget Minimalists** — Low everything, lowest bills, short tenure, highest support calls. Deal-seekers who call support often hunting for cheaper plans.
- **Power Users** — High across the board, highest bills, long tenure, lowest support calls. Most valuable, most satisfied subscribers.

### 💡 Insight

Four clusters, four personalities, four different strategies. Data Junkies need bigger data plans. Traditional Callers need international bundles. Budget Minimalists need simple cheap plans with less support friction. Power Users need VIP treatment to keep them loyal.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
colors = ["#214073", "#008C8C", "#E6592D", "#F5A623"]
for i, feat in enumerate(X.columns):
    ax = axes[i // 4, i % 4]
    df.groupby("cluster")[feat].mean().plot(
        kind="bar", ax=ax, color=colors)
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel("")
plt.suptitle("Average Feature Values by Cluster", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

### 💬 Discussion 3: Amit's Marketing Brief

*Instructor leads — students respond:*

- For EACH of the 4 segments, suggest a one-line marketing action Amit should take
- Example: "Data Junkies → Offer unlimited data plan upgrade at ₪20/month premium"
- Think about: what does each group VALUE? What keeps them as a customer? What would make them switch to a competitor?

**Takeaway**:
- **Data Junkies** → Offer unlimited data plan, promote TeleBari app features, partner with streaming services
- **Traditional Callers** → International calling bundles, family plans, avoid pushing app-only features
- **Budget Minimalists** → Simple low-cost plans, reduce support friction with FAQ/chatbot, loyalty rewards for staying >6 months
- **Power Users** → VIP support line, early access to new features, "thank you" perks — they're the most valuable, don't let a competitor poach them

---
## 🔬 Section 5: PCA — Dimensionality Reduction
*8 features → 2 dimensions. See all 500 subscribers on one plot.*

---

### 📖 PCA Recap

The 2D scatter plot in Section 2 showed only 2 of 8 features — a partial view. Some clusters that overlap in 2D may be clearly separated in the full 8D space. PCA compresses all 8 features into 2 "super-features" (principal components) that capture the most variance. Then we can plot EVERYTHING on one chart.

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)     # fit + transform in one step

print(f"Original shape: {X_scaled.shape}")
print(f"PCA shape:      {X_pca.shape}")
print(f"\nPC1 explains: {pca.explained_variance_ratio_[0]:.1%}")
print(f"PC2 explains: {pca.explained_variance_ratio_[1]:.1%}")
print(f"Total:        {sum(pca.explained_variance_ratio_):.1%}")

### ⚠️ Common Mistake

Even though 2 components capture a large share of the variance, we're still losing some information — features that don't align with PC1 or PC2 get compressed away. For ANALYSIS (not just visualization), use more components — we'll check how many we need next.

In [ ]:
pca_full = PCA().fit(X_scaled)
cumvar = pca_full.explained_variance_ratio_.cumsum()

plt.plot(range(1, len(cumvar) + 1), cumvar, "bo-", linewidth=2)
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Variance Explained")
plt.axhline(y=0.80, color="r", linestyle="--", label="80% threshold")
plt.title("How Many Components Do We Need?")
plt.legend()
plt.xticks(range(1, len(cumvar) + 1))
plt.show()

print("Cumulative variance by number of components:")
for i, cv in enumerate(cumvar, 1):
    print(f"  {i} PCs: {cv:.1%}")

2 components are enough for visualization. For deeper analysis, you'd use more — the cumulative variance plot helps you decide how many.

In [ ]:
loadings = pd.DataFrame(
    pca.components_,
    columns=X.columns,
    index=["PC1", "PC2"]
).round(2)

print("PCA Loadings (what each component measures):\n")
print(loadings.to_string())

### 💬 Discussion 4: Interpret the Components

*Instructor leads — students respond:*

- Look at the loadings table
- **PC1**: Which features have the largest positive loadings? Which have the largest negative loadings? Give PC1 a business name (e.g., "Usage Intensity" or "Spending Power")
- **PC2**: Same — which features dominate? Give PC2 a business name (e.g., "Digital vs. Traditional")
- Why does this make business sense for a telecom company?

**Takeaway**: **PC1** loads positively on `monthly_minutes`, `monthly_sms`, `monthly_bill_nis`, `account_age_months`, `international_minutes` and negatively on `support_calls` — it's an **"Overall Usage / Value"** axis. **PC2** loads positively on `monthly_data_gb` and `app_sessions_per_week` — it's a **"Digital vs. Traditional"** axis. The two fundamental axes of telecom behavior: how MUCH someone uses, and HOW they use it.

### 💡 Insight

PCA didn't just compress numbers — it discovered the two fundamental axes of telecom customer behavior: how much they use (value) and how they use it (digital vs. traditional). These are the same dimensions Amit's team intuitively thinks about.

---
## 🗺️ Section 6: The Full Picture
*Clusters + PCA = all 500 subscribers on one chart, colored by segment.*

---

In [ ]:
plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1],
                      c=df["cluster"], cmap="Set2", alpha=0.6,
                      edgecolor="gray", s=50)
plt.xlabel("PC1 — Overall Usage / Value →")
plt.ylabel("PC2 — Digital ↑ vs. Traditional ↓")
plt.title("TeleBari Customer Segments in PCA Space")
plt.colorbar(scatter, label="Cluster")
plt.show()

In [ ]:
cluster_names = {0: "Data Junkies", 1: "Power Users",
                 2: "Budget Min.", 3: "Trad. Callers"}
plt.figure(figsize=(10, 7))
for cid, name in cluster_names.items():
    mask = df["cluster"] == cid
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], alpha=0.5, label=name, s=50, edgecolor="gray")
    cx, cy = X_pca[mask, 0].mean(), X_pca[mask, 1].mean()
    plt.annotate(name, (cx, cy), fontsize=11, fontweight="bold", ha="center", va="center",
                 bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8))
plt.xlabel("PC1 — Overall Usage / Value →")
plt.ylabel("PC2 — Digital ↑ vs. Traditional ↓")
plt.title("TeleBari — 4 Customer Segments (PCA Visualization)")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

### 📖 Reading the Plot

This single chart shows all 500 subscribers, all 8 features (compressed into 2 PCA axes), and all 4 segments. Power Users are on the far right (highest overall usage). Data Junkies are in the upper area (high digital). Traditional Callers are in the lower area (high voice, low digital). Budget Minimalists are on the left (low everything).

### 💬 Discussion 5: Spot the Overlap

*Instructor leads — students respond:*

- Look at the PCA scatter plot. Which two clusters overlap the most?
- Why might they overlap in 2D but be separated in the full 8D space?
- If you were Amit and these two clusters overlapped, would you merge them into one segment or keep them separate? Why?

**Takeaway**: Clusters that overlap in 2D may be separated on features that PC1 and PC2 don't fully capture — 2 components don't explain 100% of the variance. Keep them separate — the marketing actions are completely different. The 2D plot is for **visualization**, not for decision-making. Always go back to the profile table for business decisions.

### 🎯 Exercise 1: The Full Pipeline

- Write the complete clustering + PCA pipeline as a single code block (without running K-selection — assume K=4)
- The pipeline should: (1) scale features, (2) run K-Means with K=4, (3) run PCA with 2 components, (4) create the annotated scatter plot
- This is the "recipe" you'd reuse on any new dataset

In [ ]:
##### Your code goes here #####

# Step 1: Scale

# Step 2: K-Means (K=4)

# Step 3: PCA (2 components)

# Step 4: Scatter plot colored by cluster, labeled axes

###### End of Your Code ######

### ✅ Solution

<details>
<summary>Click to reveal</summary>

```python
# --- Full Clustering + PCA Pipeline ---

# Step 1: Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Step 2: K-Means clustering
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)

# Step 3: PCA for visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Step 4: Scatter plot
plt.figure(figsize=(10, 7))
plt.scatter(X_pca[:, 0], X_pca[:, 1],
            c=labels, cmap="Set2", alpha=0.6,
            edgecolor="gray", s=50)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%} variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%} variance)")
plt.title("Customer Segments in PCA Space")
plt.colorbar(label="Cluster")
plt.show()
```

</details>

---
## 📊 Section 7: Amit's Strategy Report
*Turn analysis into action. What does Amit tell the board?*

---

### 💬 Discussion 6: Amit's Board Presentation

*Instructor leads — students respond. Amit has 3 minutes at the board meeting:*

1. **How many segments?** Why 4? (Reference both elbow and silhouette)
2. **Who are they?** Name each segment and its size. One sentence per segment.
3. **The PCA insight**: What are the two axes of customer behavior?
4. **The #1 action**: If Amit can only do ONE thing based on this analysis, what should it be?
5. **A caveat**: What should Amit warn the board about? (Hint: clusters are hypotheses, not facts)

**Takeaway**:
1. **4 segments** — both elbow and silhouette peak at K=4, each with a distinct profile.
2. Data Junkies (~130), Traditional Callers (~120), Budget Minimalists (~150), Power Users (~100).
3. PC1 = overall usage intensity, PC2 = digital vs. traditional communication style.
4. Launch a VIP retention program for Power Users — they're the highest-value segment and losing even a few costs more than the other segments combined.
5. These segments come from 3 months of data, not customer interviews. K-Means will ALWAYS find K groups — even in random noise. The segments make business sense, but they're a starting hypothesis, not ground truth.

---

### 🎓 What You Just Did

- Loaded 500 subscriber records and explored usage patterns with EDA
- Scaled features and ran K-Means clustering (no target variable!)
- Chose K=4 using the Elbow method and Silhouette score
- Profiled and named 4 customer segments with business meaning
- Used PCA to compress 8 features into 2 interpretable dimensions
- Visualized all 500 subscribers on one labeled scatter plot
- Wrote a board-ready segmentation strategy

### The Unsupervised Learning Toolkit

```python
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)          # scale first!

kmeans = KMeans(n_clusters=4, random_state=42)
labels = kmeans.fit_predict(X_scaled)       # no y!

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)         # compress to 2D
```

Notice: `.fit(X)` only — no y anywhere. That's the unsupervised signature.

### 💬 One Last Thing

Every model so far had a "right answer" — churn labels, car prices, startup valuations. Today, there was no right answer. The algorithm discovered structure on its own, and YOU validated it with business knowledge. That's unsupervised learning — and it's what companies use to understand their customers at scale.